<a href="https://colab.research.google.com/github/willismax/MediaSystem-Python-Course/blob/main/12.audio-dsp/%E8%81%BD%E8%A6%8B_Sampling_%E8%88%87_Quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

3.8 動手驗證：聽見 Sampling 與 Quantization
===================================

Sampling 與 Quantization 如果只有公式會很抽象，本教材採「破壞性實驗」概念：**故意把參數降到不合理的程度，直接聽見系統壞掉。**

先使用 NumPy 建立 440 Hz 正弦波：

實驗一｜聽見 Aliasing
---------------


In [3]:
import numpy as np
import wave

def make_tone(freq=440, seconds=2.0, sample_rate=44100):
    t = np.linspace(
        0,
        seconds,
        int(sample_rate * seconds),
        endpoint=False
    )

    signal = np.sin(2 * np.pi * freq * t)

    # 使用 16-bit PCM 表示
    quantized = (signal * 32767).astype(np.int16)

    filename = f"tone_{freq}Hz_{sample_rate}sr.wav"

    with wave.open(filename, "w") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(sample_rate)
        w.writeframes(quantized.tobytes())

    return filename


print(make_tone())
print(make_tone(sample_rate=44100))
print(make_tone(sample_rate=8000))
print(make_tone(sample_rate=1000))
print(make_tone(sample_rate=500))

tone_440Hz_44100sr.wav



440 Hz 的聲音，如果依照 Nyquist 原則，Sample Rate 至少需要高於：

```text
880 Hz
```

依序產生：

```python
make_tone(sample_rate=44100)
make_tone(sample_rate=8000)
make_tone(sample_rate=1000)
make_tone(sample_rate=500)
```

前面幾個取樣率仍高於 880 Hz，而 `500 Hz` 已經低於需求。這時候取樣得到的資料無法正確描述原本的 440 Hz 正弦波，可能呈現成完全不同的低頻訊號。

觀察重點不是「音質比較差」這麼簡單，而是：

> **系統記錄到的是另一個錯誤的訊號。**

這就是 Aliasing 最重要的地方。

In [7]:
from IPython.display import Audio

file_names = [
    'tone_440Hz_44100sr.wav',
    'tone_440Hz_8000sr.wav',
    'tone_440Hz_1000sr.wav',
    'tone_440Hz_500sr.wav'
]

for filename in file_names:
    print(f"Playing {filename}:")
    display(Audio(filename))


Playing tone_440Hz_44100sr.wav:


Playing tone_440Hz_8000sr.wav:


Playing tone_440Hz_1000sr.wav:


Playing tone_440Hz_500sr.wav:


實驗二｜聽見 Quantization Error
-------------------------



這次保留 16-bit WAV 格式，但是故意降低有效量化精度，例如只保留 8、4、2 bits 的資訊，再放回 16-bit 容器播放。

可以使用位元位移模擬：

```python
def reduce_bit_depth(signal16, effective_bits):
    shift = 16 - effective_bits
    return (signal16 >> shift) << shift
```

依序比較：

```text
16 bit
8 bit
4 bit
2 bit
```

當有效 Bit Depth 大幅下降時，波形能使用的數值層級越來越少，量化誤差會逐漸變得可以聽見。

這個實驗的重點是：

```text
Sampling 沒有改
↓
時間點數量相同

Quantization 改變
↓
每個 Sample 的精度下降
```

因此可以很清楚地區分兩者。


In [8]:
import numpy as np
import wave
from IPython.display import Audio

def reduce_bit_depth(signal16, effective_bits):
    if effective_bits >= 16:
        return signal16
    shift = 16 - effective_bits
    return (signal16 >> shift) << shift

freq = 440
seconds = 2.0
sample_rate = 44100 # Keep a high sample rate to focus on quantization

# Recreate the original 16-bit signal data (from the make_tone function logic)
# This avoids dependencies on previously saved files for the base signal.
t = np.linspace(0, seconds, int(sample_rate * seconds), endpoint=False)
signal_float = np.sin(2 * np.pi * freq * t)
original_signal_16bit = (signal_float * 32767).astype(np.int16)

def save_wav_with_bit_depth(signal_array, sample_rate, effective_bits):
    filename = f"tone_{freq}Hz_sr{sample_rate}_bit{effective_bits}.wav"
    with wave.open(filename, "w") as w:
        w.setnchannels(1)
        w.setsampwidth(2)  # Always save as 16-bit WAV container
        w.setframerate(sample_rate)
        w.writeframes(signal_array.tobytes())
    return filename

print("實驗二｜聽見 Quantization Error")
print("---------------------------------------")

# List to hold filenames for playing
audio_files_to_play = []

# Generate and save audio for different effective bit depths
for bits in [16, 8, 4, 2]:
    print(f"正在生成 {bits}-bit 有效位元深度音訊...")
    if bits == 16:
        # For 16 bits, use the original 16-bit signal directly
        processed_signal = original_signal_16bit
    else:
        # Apply bit depth reduction for less than 16 bits
        processed_signal = reduce_bit_depth(original_signal_16bit, bits)

    filename = save_wav_with_bit_depth(processed_signal, sample_rate, bits)
    audio_files_to_play.append(filename)

print("\n播放生成的音訊檔案：")
for filename in audio_files_to_play:
    print(f"播放 {filename}:")
    display(Audio(filename))


實驗二｜聽見 Quantization Error
---------------------------------------
正在生成 16-bit 有效位元深度音訊...
正在生成 8-bit 有效位元深度音訊...
正在生成 4-bit 有效位元深度音訊...
正在生成 2-bit 有效位元深度音訊...

播放生成的音訊檔案：
播放 tone_440Hz_sr44100_bit16.wav:


播放 tone_440Hz_sr44100_bit8.wav:


播放 tone_440Hz_sr44100_bit4.wav:


播放 tone_440Hz_sr44100_bit2.wav:


# 實驗小結



單純聽「逼～」的聲音時，兩者的差異在於「音調（頻率）有沒有跑掉」**與**「音質純不純（雜音多寡）」。

**實驗一｜聽見 Aliasing（混疊）**

* **聽覺特徵**：在 Sample Rate 足夠（如 44.1 kHz、8000 Hz、1000 Hz）時，聽到的都是標準的 440 Hz 音高（標準音 A）。當取樣率降到 500 Hz 時，由於低於 Nyquist 頻率門檻（440 × 2 = 880 Hz），系統產生的不再是 440 Hz，而是混疊後的 **60 Hz 低頻「逼～」**（$\vert{}500 - 440\vert{} = 60$ Hz）。
* **教學解釋**：取樣率不足時，電腦不是單純讓「音質變差」，而是直接**把訊號誤認成完全不同的頻率**。

**實驗二｜聽見 Quantization Error（量化誤差）**

* **聽覺特徵**：音高（440 Hz）始終沒有改變，但在 16-bit 下是平滑乾淨的純單音；降到 8-bit、4-bit、2-bit 時，原本圓滑的正弦波被切成階梯狀（接近方波/鋸齒波），「逼～」聲中會混入越來越明顯的**沙沙雜音、毛邊與金屬毛刺感**（8-bit 復古遊戲音效感）。
* **教學解釋**：取樣點時間完全沒變，但每次能記錄的數值精度大幅降低，數值誤差轉化為**量化雜訊與諧波失真**。

| 實驗維度 | 實驗一：Aliasing | 實驗二：Quantization Error |
| --- | --- | --- |
| **改變的參數** | 取樣率（Sample Rate） | 位元深度（Bit Depth） |
| **音高（頻率）** | **改變**（直接變成別的音） | **不變**（依然是 440 Hz） |
| **音質（波形）** | 乾淨，但頻率錯誤 | 出現明顯雜音、毛邊與失真 |
| **結論** | 「有沒有量到原本的節奏」 | 「量到的數值記得到底準不準」 |